In [17]:
%load_ext autoreload
%autoreload 2

import jax.numpy as jnp
from mGST.utility_functions_comparisons import get_mgst_tensors_from_psd_representation, create_4q_gst_config
from iqm.benchmarks.compressive_gst.gst_analysis import dataset_counts_to_mgst_format
import pickle as pkl
import random


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Optimized data

In [3]:
gate_set_psd = jnp.load("data/optimization_results_4q_10000seq_500shots_rank_4.npz")
gate_set_psd.keys()

KeysView(NpzFile 'data/optimization_results_4q_10000seq_500shots_rank_4.npz' with keys: kraus_tensor, povm_psd, state_psd, cost_fn_history)

In [5]:
kraus_tensor = gate_set_psd["kraus_tensor"]
povm_psd = gate_set_psd["povm_psd"]
state_psd = gate_set_psd["state_psd"]

gate_set_superop = get_mgst_tensors_from_psd_representation(
    kraus_tensor=kraus_tensor, povm_psd=povm_psd, state_psd=state_psd
)

gate_set_superop.keys()

INFO:2026-05-13 16:31:29,475:jax._src.xla_bridge:927: Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
2026-05-13 16:31:29,475 - jax._src.xla_bridge - INFO - Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:2026-05-13 16:31:29,482:jax._src.xla_bridge:927: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/emiliano.godinez/.pyenv/versions/3.11.10/lib/libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Users/emiliano.godinez/.pyenv/versions/3.11.10/lib/libtpu.so' (no such file), '/opt/homebrew/lib/libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/lib/libtpu.so' (no such file), '/Users/emiliano.godinez/.pyenv/versions/3.11.10/lib/libtpu.so' (no such file), '/System/Volumes/Preboot/

dict_keys(['kraus', 'povm', 'state'])

In [7]:
jnp.savez(
    "data/ops_reconstructed_mgst_4q_rank_4_all.npz",
    kraus=gate_set_superop["kraus"],
    povm=gate_set_superop["povm"],
    state=gate_set_superop["state"],
)

# Experiment design

In [8]:
# load pkl file
with open("data/run_result_4Q_10000seq_500shots.pkl", "rb") as f:
    experiment_design = pkl.load(f)

In [9]:
type(experiment_design)

iqm.benchmarks.benchmark_definition.BenchmarkRunResult

In [10]:
benchmark_circuits = experiment_design.circuits.benchmark_circuits
transpiled_circuits = benchmark_circuits[0]
unstrapiled_circuits = benchmark_circuits[1]
transpiled_circuits.group_names


['[0, 1, 3, 4]']

In [11]:
circuit_groups_unstrapiled = unstrapiled_circuits.groups
assert len(circuit_groups_unstrapiled) == 1
circuit_groups_unstrapiled = circuit_groups_unstrapiled[0]
print(circuit_groups_unstrapiled.name, len(circuit_groups_unstrapiled.circuits))
circuit_groups_unstrapiled.qubits

[0, 1, 3, 4] 10000


{Qubit(QuantumRegister(4, 'q'), 0),
 Qubit(QuantumRegister(4, 'q'), 1),
 Qubit(QuantumRegister(4, 'q'), 2),
 Qubit(QuantumRegister(4, 'q'), 3)}

In [12]:
circuit_groups_transpiled = transpiled_circuits.groups
assert len(circuit_groups_transpiled) == 1
circuit_groups_transpiled = circuit_groups_transpiled[0]
print(circuit_groups_transpiled.name, len(circuit_groups_transpiled.circuits))
circuit_groups_transpiled.qubits

[0, 1, 3, 4] 10000


{Qubit(QuantumRegister(20, 'q'), 0),
 Qubit(QuantumRegister(20, 'q'), 1),
 Qubit(QuantumRegister(20, 'q'), 10),
 Qubit(QuantumRegister(20, 'q'), 11),
 Qubit(QuantumRegister(20, 'q'), 12),
 Qubit(QuantumRegister(20, 'q'), 13),
 Qubit(QuantumRegister(20, 'q'), 14),
 Qubit(QuantumRegister(20, 'q'), 15),
 Qubit(QuantumRegister(20, 'q'), 16),
 Qubit(QuantumRegister(20, 'q'), 17),
 Qubit(QuantumRegister(20, 'q'), 18),
 Qubit(QuantumRegister(20, 'q'), 19),
 Qubit(QuantumRegister(20, 'q'), 2),
 Qubit(QuantumRegister(20, 'q'), 3),
 Qubit(QuantumRegister(20, 'q'), 4),
 Qubit(QuantumRegister(20, 'q'), 5),
 Qubit(QuantumRegister(20, 'q'), 6),
 Qubit(QuantumRegister(20, 'q'), 7),
 Qubit(QuantumRegister(20, 'q'), 8),
 Qubit(QuantumRegister(20, 'q'), 9)}

In [13]:
# test a single circuit
idx = 560 #5607
circuit_unstrapiled = circuit_groups_unstrapiled.circuits[idx]
circuit_unstrapiled.draw(idle_wires=False)

┌────────────┐ ░     ░                ░ ┌─┐         
q_0: ┤ R(π/2,π/2) ├─░──■──░────────────────░─┤M├─────────
     └────────────┘ ░  │  ░                ░ └╥┘┌─┐      
q_1: ───────────────░──■──░────────────────░──╫─┤M├──────
                    ░     ░                ░  ║ └╥┘┌─┐   
q_2: ───────────────░──■──░────────────────░──╫──╫─┤M├───
                    ░  │  ░ ┌────────────┐ ░  ║  ║ └╥┘┌─┐
q_3: ───────────────░──■──░─┤ R(π/2,π/2) ├─░──╫──╫──╫─┤M├
                    ░     ░ └────────────┘ ░  ║  ║  ║ └╥┘
c: 4/═════════════════════════════════════════╩══╩══╩══╩═
                                              0  1  2  3

In [129]:

rnd_numbers = random.sample(range(len(circuit_groups_transpiled.circuits)), 10)
print(f"Randomly selected circuit indices: {rnd_numbers}")
for idx in rnd_numbers:
    circuit_transpiled = circuit_groups_transpiled.circuits[idx]
    print(circuit_transpiled.draw(idle_wires=False))

Randomly selected circuit indices: [9600, 8862, 4179, 1119, 345, 2081, 978, 5031, 9455, 2231]
                        ░     ░     ░     ░              ░                ░ »
q_0 -> 0 ───────────────░──■──░──■──░──■──░──────────────░────────────────░─»
                        ░  │  ░  │  ░  │  ░              ░ ┌────────────┐ ░ »
q_1 -> 1 ───────────────░──■──░──■──░──■──░──────────────░─┤ R(π/2,π/2) ├─░─»
                        ░     ░     ░     ░ ┌──────────┐ ░ └────────────┘ ░ »
q_3 -> 3 ───────────────░──■──░──■──░──■──░─┤ R(π/2,0) ├─░────────────────░─»
         ┌────────────┐ ░  │  ░  │  ░  │  ░ └──────────┘ ░                ░ »
q_4 -> 4 ┤ R(π/2,π/2) ├─░──■──░──■──░──■──░──────────────░────────────────░─»
         └────────────┘ ░     ░     ░     ░              ░                ░ »
    c: 4/═══════════════════════════════════════════════════════════════════»
                                                                            »
«                      ░     ░ ┌────────────┐ ░ 

In [26]:
import qiskit.qpy as qpy
# now saving them
transpiled_circuits = circuit_groups_transpiled.circuits
untranspiled_circuits = circuit_groups_unstrapiled.circuits
# Save
with open("data/transpiled_circuits.qpy", "wb") as f:
    qpy.dump(transpiled_circuits, f)
    
with open("data/untranspiled_circuits.qpy", "wb") as f:
    qpy.dump(untranspiled_circuits, f)

# Counts

In [19]:
dataset = experiment_design.dataset

In [20]:
type(dataset)

xarray.core.dataset.Dataset

In [21]:
qubit_layout = [0, 1, 3, 4]
prob_matrix = dataset_counts_to_mgst_format(dataset=dataset, qubit_layout=qubit_layout)
prob_matrix.shape

(16, 10000)

# Circuits

In [142]:
max_circuits_per_batch = 100 # limit for real backend. Update: I had to change this because it was throwing errors.
num_sequences = 10000 # >= 3 x total_params
shots = 500
kraus_rank = 4
config_4q = create_4q_gst_config(kraus_rank=kraus_rank, num_gate_sequences=num_sequences, shots=shots, max_gates_per_batch=None, max_circuits_per_batch=max_circuits_per_batch, seq_len_list=[1, 10, 19])
config_4q.gate_labels

['Rx(pi/2)',
 'Rx(pi/2)',
 'Rx(pi/2)',
 'Rx(pi/2)',
 'Ry(pi/2)',
 'Ry(pi/2)',
 'Ry(pi/2)',
 'Ry(pi/2)',
 'CZ-CZ']

In [7]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import CZGate, RGate

cz_cz = QuantumCircuit(4)
cz_cz.append(CZGate(), [0,1])
cz_cz.append(CZGate(), [2,3])

gate_list = [
    RGate(0.5 * jnp.pi, 0),
    RGate(0.5 * jnp.pi, 0),
    RGate(0.5 * jnp.pi, 0),
    RGate(0.5 * jnp.pi, 0),
    RGate(0.5 * jnp.pi, jnp.pi / 2),
    RGate(0.5 * jnp.pi, jnp.pi / 2),
    RGate(0.5 * jnp.pi, jnp.pi / 2),
    RGate(0.5 * jnp.pi, jnp.pi / 2),
    cz_cz,
]
gates = [QuantumCircuit(4, 0) for _ in range(len(gate_list))]
gate_qubits = [[0], [1], [2], [3], [0], [1], [2], [3], [0, 1, 2, 3]]
for i, gate in enumerate(gate_list):
    if isinstance(gate, QuantumCircuit):
        gates[i].compose(gate, gate_qubits[i], inplace = True)
    else:
        gates[i].append(gate, gate_qubits[i])
        
gate_labels = ["Rx(pi/2)", "Rx(pi/2)", "Rx(pi/2)", "Rx(pi/2)", 
                "Ry(pi/2)", "Ry(pi/2)", "Ry(pi/2)", "Ry(pi/2)", 
                "CZ-CZ"]

for gate in gates:
    print(gate.draw())

     ┌──────────┐
q_0: ┤ R(π/2,0) ├
     └──────────┘
q_1: ────────────
                 
q_2: ────────────
                 
q_3: ────────────
                 
                 
q_0: ────────────
     ┌──────────┐
q_1: ┤ R(π/2,0) ├
     └──────────┘
q_2: ────────────
                 
q_3: ────────────
                 
                 
q_0: ────────────
                 
q_1: ────────────
     ┌──────────┐
q_2: ┤ R(π/2,0) ├
     └──────────┘
q_3: ────────────
                 
                 
q_0: ────────────
                 
q_1: ────────────
                 
q_2: ────────────
     ┌──────────┐
q_3: ┤ R(π/2,0) ├
     └──────────┘
     ┌────────────┐
q_0: ┤ R(π/2,π/2) ├
     └────────────┘
q_1: ──────────────
                   
q_2: ──────────────
                   
q_3: ──────────────
                   
                   
q_0: ──────────────
     ┌────────────┐
q_1: ┤ R(π/2,π/2) ├
     └────────────┘
q_2: ──────────────
                   
q_3: ──────────────
            